In [0]:
# =============================================================================
# BRONZE LAYER — TTC Stops and Routes Ingestion
# Toronto Chronic Congestion Root-Cause Intelligence Platform
#
# What this notebook does:
#   - Reads stops.txt and routes.txt from the /data folder
#   - These are GTFS files — they look like CSV files and are read the same way
#   - Adds an ingestion_timestamp column to each
#   - Writes both as raw Bronze Delta tables — NO cleaning, NO filtering
#
# Tables created:
#   - bronze.ttc_stops    (stop_id, stop_name, stop_lat, stop_lon, etc.)
#   - bronze.ttc_routes   (route_id, route_short_name, route_type, etc.)
#
# What is GTFS?
#   GTFS (General Transit Feed Specification) is the global standard format
#   for transit data. Every transit agency in the world publishes their
#   stops, routes, and schedules in this format. The files are .txt but
#   they are comma-separated — Spark reads them exactly like CSV files.
#
# Why only stops.txt and routes.txt?
#   - stops.txt  → tells us WHERE each TTC stop is (lat/lon)
#   - routes.txt → tells us WHAT TYPE each route is (streetcar, subway, bus)
#   - We join these two in Silver to know: "which stop belongs to which route type"
#   - stop_times.txt (198MB) has schedule timing — too large and not needed
#   - All other GTFS files are also not needed for this project
#
# Run this manually once to create the initial tables.
# After that, the Databricks Workflow will call it nightly.
# =============================================================================


# -----------------------------------------------------------------------------
# STEP 1 — Import libraries
# -----------------------------------------------------------------------------
# pyspark.sql.functions gives us current_timestamp() for the ingestion stamp

from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# STEP 2 — Define file paths
# -----------------------------------------------------------------------------
# Update the email below to match YOUR Databricks account email

REPO_BASE = "/Workspace/Repos/panditadvait75@gmail.com/toronto-traffic-pipeline"

STOPS_PATH  = f"{REPO_BASE}/data/stops.txt"
ROUTES_PATH = f"{REPO_BASE}/data/routes.txt"


# -----------------------------------------------------------------------------
# STEP 3 — Read stops.txt into a Spark DataFrame
# -----------------------------------------------------------------------------
# Even though the file extension is .txt, it is comma-separated with a header.
# We read it exactly like a CSV file — Spark does not care about the extension.
#
# inferSchema=True → Spark detects column types automatically
#   stop_lat and stop_lon will be detected as doubles (decimal numbers)
#   stop_id, stop_name etc. will be detected as strings
#   This is fine for Bronze — Silver will validate and clean types
#
# header=True → First row is column names, not data

print("Reading stops.txt...")

df_stops = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(STOPS_PATH)
)

print(f"stops.txt row count: {df_stops.count()}")
print("stops.txt columns:")
df_stops.printSchema()


# -----------------------------------------------------------------------------
# STEP 4 — Add ingestion_timestamp to stops
# -----------------------------------------------------------------------------

df_stops = df_stops.withColumn("ingestion_timestamp", F.current_timestamp())


# -----------------------------------------------------------------------------
# STEP 5 — Write stops to Bronze Delta table
# -----------------------------------------------------------------------------
# mode("overwrite") → Replace the entire table each run
#
# Why overwrite for TTC stops?
# TTC publishes a full updated GTFS feed when stops change.
# The full current list of stops replaces the old list.
# There is no partial update — it is always a full snapshot.
# So overwriting the Bronze table each run is correct.
#
# No partitioning needed — stops.txt is a small reference table
# (TTC has roughly 12,000 stops in Toronto — very manageable)

print("Writing bronze.ttc_stops...")

(
    df_stops
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.ttc_stops")
)

print("bronze.ttc_stops written successfully.")


# -----------------------------------------------------------------------------
# STEP 6 — Read routes.txt into a Spark DataFrame
# -----------------------------------------------------------------------------
# routes.txt is even smaller — TTC has roughly 150-200 routes total.
# The most important column is route_type:
#   0 = Streetcar  → runs on road, mixed with traffic, biggest congestion impact
#   1 = Subway     → underground, does not affect road traffic directly
#   3 = Bus        → runs on road, significant congestion impact
# We will use route_type heavily in Silver and Gold for congestion scoring.

print("Reading routes.txt...")

df_routes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ROUTES_PATH)
)

print(f"routes.txt row count: {df_routes.count()}")
print("routes.txt columns:")
df_routes.printSchema()


# -----------------------------------------------------------------------------
# STEP 7 — Add ingestion_timestamp to routes
# -----------------------------------------------------------------------------

df_routes = df_routes.withColumn("ingestion_timestamp", F.current_timestamp())


# -----------------------------------------------------------------------------
# STEP 8 — Write routes to Bronze Delta table
# -----------------------------------------------------------------------------
# Same overwrite logic as stops — full snapshot replacement each run.

print("Writing bronze.ttc_routes...")

(
    df_routes
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.ttc_routes")
)

print("bronze.ttc_routes written successfully.")


# -----------------------------------------------------------------------------
# STEP 9 — Validation
# -----------------------------------------------------------------------------

print("\n--- VALIDATION ---")

stops_count = spark.sql("SELECT COUNT(*) as row_count FROM bronze.ttc_stops").collect()[0]["row_count"]
print(f"bronze.ttc_stops row count: {stops_count}")

routes_count = spark.sql("SELECT COUNT(*) as row_count FROM bronze.ttc_routes").collect()[0]["row_count"]
print(f"bronze.ttc_routes row count: {routes_count}")

print("\nStops preview (first 5 rows):")
spark.sql("SELECT * FROM bronze.ttc_stops LIMIT 5").display()

print("\nRoutes preview (first 5 rows):")
spark.sql("SELECT * FROM bronze.ttc_routes LIMIT 5").display()

# Very useful check — confirm we have all 3 route types
# 0 = Streetcar, 1 = Subway, 3 = Bus
print("\nRoute type breakdown (0=Streetcar, 1=Subway, 3=Bus):")
spark.sql("""
    SELECT route_type, COUNT(*) as count
    FROM bronze.ttc_routes
    GROUP BY route_type
    ORDER BY route_type
""").display()

print("\nBronze ingestion complete for Dataset 3 — TTC Stops and Routes.")